## LoCoMo benchmark (conversation → memory → QA → metrics)

This section evaluates **FUMemory / STMemory / LTMemory / MBMemory / GAMemory / SCMemory / MGMemory / RFMemory / MTMemory** on the real `locomo10.json` benchmark.

Pipeline per sample:

- Load `conversation` (all sessions) and store into memory
- For each QA question: `recall(question)` → build `context` → call an LLM to answer
- Compute metrics vs the gold answer: **F1**, **ROUGE-L (RL)**, **BLEU-1/2 (B1/B2)**, **METEOR**, **BERTScore-F1 (BERTF1)**, **embedding cosine similarity (Sim)**
- Run **LLM-judge** scoring using the same OpenRouter model config used above (`arcee-ai/trinity-large-preview:free`)


In [1]:
import os
from dotenv import load_dotenv
import memengine

load_dotenv()
_exports = ("FUMemory", "STMemory", "LTMemory", "MBMemory", "GAMemory", "SCMemory", "MGMemory", "RFMemory", "MTMemory", "MemoryConfig")
print("memengine:", memengine)
print("memengine exports:", [n for n in _exports if hasattr(memengine, n)])

from memengine import (
    MemoryConfig,
    FUMemory,
    STMemory,
    LTMemory,
    MBMemory,
    GAMemory,
    SCMemory,
    MGMemory,
    RFMemory,
    MTMemory,
)

d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


memengine: <module 'memengine' from 'd:\\Anaconda\\envs\\py312pt291cu128\\Lib\\site-packages\\memengine\\__init__.py'>
memengine exports: ['FUMemory', 'STMemory', 'LTMemory', 'MBMemory', 'GAMemory', 'SCMemory', 'MGMemory', 'RFMemory', 'MTMemory', 'MemoryConfig']


In [2]:
def make_common_config(*, usable_gpu: str = "", display_method: str = "ScreenDisplay") -> dict:
    # Minimal, runnable configs derived from the official open-source defaults,
    # but adapted to run locally without external model paths.
    return {
        "global_config": {"usable_gpu": usable_gpu},
        "storage": {},
        "display": {
            "method": display_method,
            "prefix": "----- Current Memory Start (%s) -----",
            "suffix": "----- Current Memory End -----",
            "key_format": "(%s)",
            "key_value_sep": "\n",
            "item_sep": "\n",
            # FileDisplay-only arg (ignored by ScreenDisplay)
            "output_path": "logs/sample.log",
        },
        "recall": {
            "truncation": {
                "method": "LMTruncation",
                "mode": "word",
                "number": 256,
                "path": "",  # only used for token-based truncation
            },
            "utilization": {
                "method": "ConcateUtilization",
                "prefix": "[Memory Start]",
                "suffix": "[Memory End]",
                "list_config": {"index": True, "sep": "\n"},
                "dict_config": {"key_format": "(%s)", "key_value_sep": "\n", "item_sep": "\n"},
            },
            "empty_memory": "None",
        },
        "store": {},
    }


def make_text_retrieval_config(*, topk: int = 5, st_model: str = "sentence-transformers/all-MiniLM-L6-v2") -> dict:
    # LTMemory / MBMemory rely on embeddings; this uses SentenceTransformers.
    return {
        "method": "TextRetrieval",
        "encoder": {
            "method": "STEncoder",
            "name": st_model,
            "dimension": 384,
            "path": st_model,
        },
        "mode": "cosine",
        "topk": topk,
    }


def run_memory(
    memory,
    obs_list,
    query_text,
    *,
    with_time: bool = False,
    fixed_time: int | None = None,
    time_bucket: int = 1,
):
    memory.reset()
    for i, obs in enumerate(obs_list):
        if with_time:
            if fixed_time is not None:
                t = fixed_time
            else:
                tb = max(1, int(time_bucket))
                t = i // tb
            memory.store({"text": obs, "time": t})
        else:
            memory.store(obs)
    return memory.recall(query_text)

In [3]:
import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

LOCOMO_PATH = Path("locomo10_simplified.json")
assert LOCOMO_PATH.exists(), f"Not found: {LOCOMO_PATH.resolve()}"

locomo_raw = json.loads(LOCOMO_PATH.read_text(encoding="utf-8"))
print("locomo items:", len(locomo_raw))
print("first keys:", list(locomo_raw[0].keys()))


def _iter_sessions(conv: dict) -> Iterable[Tuple[int, str, List[dict]]]:
    """Yield (session_idx, session_date_time, session_turns)."""
    # Keys are like: session_1_date_time, session_1, session_2_date_time, session_2, ...
    idx = 1
    while True:
        dt_key = f"session_{idx}_date_time"
        s_key = f"session_{idx}"
        if dt_key not in conv or s_key not in conv:
            break
        yield idx, conv[dt_key], conv[s_key]
        idx += 1


def flatten_conversation(sample: dict) -> List[dict]:
    """Flatten LoCoMo sample['conversation'] into ordered turns.

    Each turn dict contains at least: text, speaker, dia_id, session_idx, session_dt.
    """
    conv = sample["conversation"]
    turns: List[dict] = []
    for session_idx, session_dt, sess in _iter_sessions(conv):
        for t in sess:
            if "text" not in t:
                continue
            turns.append({
                "session_idx": session_idx,
                "session_dt": session_dt,
                "speaker": t.get("speaker", ""),
                "dia_id": t.get("dia_id", ""),
                "text": t["text"],
                # keep optional multimodal fields (won't always exist)
                "img_url": t.get("img_url"),
                "blip_caption": t.get("blip_caption"),
                "query": t.get("query"),
            })
    return turns


def format_turn_for_memory(turn: dict) -> str:
    parts = []
    dia = turn.get("dia_id")
    spk = turn.get("speaker")
    txt = turn.get("text")
    if dia:
        parts.append(f"{dia}")
    if spk:
        parts.append(f"{spk}")
    header = " | ".join(parts) if parts else "Turn"

    body = txt
    if turn.get("blip_caption"):
        body += f"\n[image_caption] {turn['blip_caption']}"
    if turn.get("query"):
        body += f"\n[image_query] {turn['query']}"

    return f"{header}: {body}".strip()


# quick structure check
_turns0 = flatten_conversation(locomo_raw[0])
print("sample0 turns:", len(_turns0))
print("first turn:", format_turn_for_memory(_turns0[0])[:120] + ("..." if len(format_turn_for_memory(_turns0[0])) > 120 else ""))


locomo items: 10
first keys: ['event_summary', 'observation', 'session_summary', 'sample_id', 'conversation', 'qa']
sample0 turns: 18
first turn: D1:1 | Caroline: Hey Mel! Good to see you! How have you been?


In [4]:
# --- LLM client (OpenRouter via OpenAI-compatible API) ---

# We will use the same model config as the MBMemory summarizer above for LLM-judge.
JUDGE_LLM_CONFIG = {
    "name": os.environ.get("OPENROUTER_MODEL", "arcee-ai/trinity-large-preview:free"),
    "api_key": os.environ.get("OPENROUTER_API_KEY", ""),
    "base_url": os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    "temperature": 0.0,
}
assert JUDGE_LLM_CONFIG["api_key"], "Missing env: OPENROUTER_API_KEY"

# Answering model: default to the same (override if you want)
ANSWER_LLM_CONFIG = dict(JUDGE_LLM_CONFIG)


def _openai_client(base_url: str, api_key: str):
    from openai import OpenAI
    return OpenAI(base_url=base_url, api_key=api_key)


def chat_completion(*, llm_cfg: dict, messages: List[dict], max_tokens: int = 256) -> str:
    client = _openai_client(llm_cfg["base_url"], llm_cfg["api_key"])
    resp = client.chat.completions.create(
        model=llm_cfg["name"],
        messages=messages,
        temperature=float(llm_cfg.get("temperature", 0.0)),
        max_tokens=max_tokens,
    )
    return (resp.choices[0].message.content or "").strip()


def answer_question(*, question: str, context: str, llm_cfg: dict = ANSWER_LLM_CONFIG) -> str:
    sys_msg = (
        "You answer questions using ONLY the provided context. "
        "If the answer is not in the context, reply with 'Unknown'. "
        "Keep the answer short and specific."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    return chat_completion(
        llm_cfg=llm_cfg,
        messages=[{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}],
        max_tokens=128,
    )


def llm_judge(*, question: str, gold: str, pred: str, context: str, llm_cfg: dict = JUDGE_LLM_CONFIG) -> dict:
    sys_msg = "You are a strict evaluator. Output JSON only."
    user_msg = f"""Rate the prediction against the gold answer for the given question.

Return a JSON object with fields:
- score: number in [0,1]
- verdict: one of ["correct","partially_correct","incorrect","unknown"]
- rationale: short string

Question: {question}

Gold answer: {gold}

Prediction: {pred}

Retrieved context:
{context}
"""
    raw = chat_completion(
        llm_cfg=llm_cfg,
        messages=[{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}],
        max_tokens=256,
    )
    # robust JSON extraction
    m = re.search(r"\{[\s\S]*\}", raw)
    if not m:
        return {"score": 0.0, "verdict": "incorrect", "rationale": f"non_json: {raw[:200]}"}
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return {"score": 0.0, "verdict": "incorrect", "rationale": f"bad_json: {raw[:200]}"}

    score = obj.get("score", 0.0)
    try:
        score = float(score)
    except Exception:
        score = 0.0
    score = max(0.0, min(1.0, score))

    verdict = obj.get("verdict", "incorrect")
    rationale = obj.get("rationale", "")
    return {"score": score, "verdict": verdict, "rationale": rationale}


In [5]:
# --- Metrics ---

from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

try:
    from bert_score import score as bert_score
except Exception as e:
    raise RuntimeError("bert-score package missing; run the install cell above") from e

try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("sentence-transformers missing; run the install cell above") from e


def _normalize_text(s: Any) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    return s.strip()


def _tokenize(s: str) -> List[str]:
    s = _normalize_text(s).lower()
    return re.findall(r"[a-z0-9]+", s)


def token_f1(pred: str, gold: str) -> float:
    p = _tokenize(pred)
    g = _tokenize(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    from collections import Counter

    pc = Counter(p)
    gc = Counter(g)
    common = pc & gc
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(p)
    recall = num_same / len(g)
    return 2 * precision * recall / (precision + recall)


_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
_smooth = SmoothingFunction().method1


def rouge_l_f(pred: str, gold: str) -> float:
    scores = _rouge.score(_normalize_text(gold), _normalize_text(pred))
    return float(scores["rougeL"].fmeasure)


def bleu_1_2(pred: str, gold: str) -> Tuple[float, float]:
    ref = [_tokenize(gold)]
    hyp = _tokenize(pred)
    if not hyp or not ref[0]:
        return 0.0, 0.0
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=_smooth)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=_smooth)
    return float(b1), float(b2)


def meteor(pred: str, gold: str) -> float:
    # NLTK's meteor_score expects pre-tokenized inputs (Iterable[str]) in newer versions.
    ref = _tokenize(gold)
    hyp = _tokenize(pred)
    if not hyp and not ref:
        return 1.0
    if not hyp or not ref:
        return 0.0
    return float(meteor_score([ref], hyp))


# Embeddings for Sim (batch-friendly)
# Prefer GPU if available.
try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("[metrics] DEVICE:", DEVICE)

EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
_emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)


def cosine_sim_batch(preds: Sequence[str], golds: Sequence[str]) -> List[float]:
    p = [_normalize_text(x) for x in preds]
    g = [_normalize_text(x) for x in golds]
    a = _emb_model.encode(p, normalize_embeddings=True)
    b = _emb_model.encode(g, normalize_embeddings=True)
    return list(np.sum(a * b, axis=1).astype(float))


def bert_f1_batch(preds: Sequence[str], golds: Sequence[str], *, batch_size: int = 32) -> List[float]:
    p = [_normalize_text(x) for x in preds]
    g = [_normalize_text(x) for x in golds]
    P, R, F1 = bert_score(
        p,
        g,
        lang="en",
        verbose=False,
        device=DEVICE,
        batch_size=int(batch_size),
    )
    return [float(x) for x in F1.detach().cpu().numpy().tolist()]


[metrics] DEVICE: cuda


In [6]:
# --- Unified MemoryAdapter ---

@dataclass
class MemorySpec:
    name: str
    build: Callable[[], Any]
    store_mode: str  # "text" or "mb"
    recall_topk: int


class MemoryAdapter:
    def __init__(self, mem: Any, *, store_mode: str, time_bucket: int = 20):
        self.mem = mem
        self.store_mode = store_mode
        self.time_bucket = max(1, int(time_bucket))

    def reset(self):
        self.mem.reset()

    def store_turns(self, turns: Sequence[dict]):
        for i, t in enumerate(turns):
            text = format_turn_for_memory(t)
            if self.store_mode == "mb":
                # bucket time to control summarization cadence
                tt = i // self.time_bucket
                self.mem.store({"text": text, "time": tt})
            else:
                self.mem.store(text)

    def recall_context(self, question: str) -> str:
        ctx = self.mem.recall(question)
        return _normalize_text(ctx)


def make_memory_specs(*, topk: int = 5, st_model: str = "sentence-transformers/all-MiniLM-L6-v2") -> List[MemorySpec]:
    # Reuse config helpers defined earlier in this notebook.
    def build_fu():
        cfg = make_common_config()
        cfg["name"] = "FUMemory"
        cfg["store"] = {"method": "FUMemoryStore"}
        cfg["recall"]["method"] = "FUMemoryRecall"
        return FUMemory(MemoryConfig(cfg))

    def build_st():
        cfg = make_common_config()
        cfg["name"] = "STMemory"
        cfg["store"] = {"method": "LTMemoryStore"}
        cfg["recall"].update({
            "method": "STMemoryRecall",
            "time_retrieval": {"method": "TimeRetrieval", "mode": "raw", "topk": topk},
        })
        return STMemory(MemoryConfig(cfg))

    def build_lt():
        cfg = make_common_config()
        cfg["name"] = "LTMemory"
        cfg["store"] = {"method": "LTMemoryStore"}
        cfg["recall"].update({
            "method": "LTMemoryRecall",
            "text_retrieval": make_text_retrieval_config(topk=topk, st_model=st_model),
        })
        return LTMemory(MemoryConfig(cfg))

    def build_mb():
        cfg = make_common_config()
        cfg["name"] = "MBMemory"
        cfg["store"] = {
            "method": "MBMemoryStore",
            "summarizer": {
                "method": "LLMSummarizer",
                "LLM_config": {
                    "method": "APILLM",
                    "name": JUDGE_LLM_CONFIG["name"],
                    "api_key": JUDGE_LLM_CONFIG["api_key"],
                    "base_url": JUDGE_LLM_CONFIG["base_url"],
                    "temperature": float(JUDGE_LLM_CONFIG.get("temperature", 0.0)),
                },
                "prompt": {
                    "template": "Content: {content}\nSummarize the above content concisely, extracting the main themes and key information.",
                    "input_variables": ["content"],
                },
            },
        }
        cfg["recall"].update({
            "method": "MBMemoryRecall",
            "text_retrieval": make_text_retrieval_config(topk=topk, st_model=st_model),
        })

        mb = MBMemory(MemoryConfig(cfg))

        # Keep MBMemory robust against occasional empty outputs from free endpoints.
        class SafeSummarizer:
            def __init__(self, inner):
                self.inner = inner

            def reset(self):
                r = getattr(self.inner, "reset", None)
                if callable(r):
                    r()

            def __call__(self, content):
                for _ in range(2):
                    try:
                        s = self.inner(content)
                    except Exception:
                        s = None
                    if isinstance(s, str) and s.strip():
                        return s

                text = content if isinstance(content, str) else str(content)
                text = text.strip()
                if not text:
                    return "Summary (fallback): <empty content>"
                return "Summary (fallback): " + (text[:600] + ("..." if len(text) > 600 else ""))

        mb.store_op.summarizer = SafeSummarizer(mb.store_op.summarizer)
        return mb

    # --- GAMemory (Generative Agents): importance judge per observation ---
    def build_ga():
        cfg = make_common_config()
        cfg["name"] = "GAMemory"
        cfg["store"] = {"method": "GAMemoryStore"}
        cfg["recall"].update({
            "method": "GAMemoryRecall",
            "topk": 8,
            "text_retrieval": make_text_retrieval_config(topk=32, st_model=st_model),
            "time_retrieval": {"method": "TimeRetrieval", "mode": "exp", "coef": {"decay": 0.99}, "topk": 32},
            "importance_retrieval": {"method": "ValueRetrieval", "mode": "identical", "topk": 32},
            "importance_judge": {
                "method": "LLMJudge",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "post_scale": 10,
                "prompt": {"template": "You are scoring how important a memory is for future use.\nMemory: {message}\nReturn ONLY a number from 0 to 10.", "input_variables": ["message"]},
            },
        })
        cfg["reflect"] = {
            "reflector": {
                "threshold": 1_000_000_000,
                "reflection_topk": 5,
                "question_number": 3,
                "insight_number": 3,
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "question_prompt": {"template": "Given the information below, write exactly {question_number} questions (one per line).\n\nInformation:\n{information}", "input_variables": ["information", "question_number"]},
                "insight_prompt": {"template": "Given the statements below, write exactly {insight_number} high-level insights (one per line).\n\nStatements:\n{statements}", "input_variables": ["statements", "insight_number"]},
            }
        }
        return GAMemory(MemoryConfig(cfg))

    # --- SCMemory (Self-Controlled): flash + activation + LLM judges ---
    def build_sc():
        cfg = make_common_config()
        cfg["name"] = "SCMemory"
        cfg["store"] = {"method": "SCMemoryStore"}
        cfg["recall"].update({
            "method": "SCMemoryRecall",
            "flash_capacity": 5,
            "activation_topk": 8,
            "text_retrieval": make_text_retrieval_config(topk=32, st_model=st_model),
            "time_retrieval": {"method": "TimeRetrieval", "mode": "exp", "coef": {"decay": 0.99}, "topk": 32},
            "summarizer": {
                "method": "LLMSummarizer",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "prompt": {"template": "Summarize in 1 sentence, keep key entities and dates.\nContent: {content}", "input_variables": ["content"]},
            },
            "activation_judge": {
                "method": "LLMJudge",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "prompt": {"template": "Query: {query}\nFlash memory: {flash_memory}\nDo we need to retrieve more history beyond flash memory to answer? Return ONLY True or False.", "input_variables": ["query", "flash_memory"]},
            },
            "summary_judge": {
                "method": "LLMJudge",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "prompt": {"template": "Query: {query}\nActivation summary: {activation_summary}\nFlash memory: {flash_memory}\nIs the activation summary sufficient (no need for full texts)? Return ONLY True or False.", "input_variables": ["query", "activation_summary", "flash_memory"]},
            },
        })
        return SCMemory(MemoryConfig(cfg))

    # --- MGMemory (MemGPT-style): hierarchical + trigger + summarizer ---
    def build_mg():
        cfg = make_common_config()
        cfg["name"] = "MGMemory"
        cfg["store"] = {
            "method": "MGMemoryStore",
            "summarizer": {
                "method": "LLMSummarizer",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "prompt": {"template": "Recursive summary (keep factual details).\nPrevious summary: {recursive_summary}\nNew content: {flush_context}\nReturn updated summary only.", "input_variables": ["recursive_summary", "flush_context"]},
            },
            "flush_checker": {"method": "LMTruncation", "mode": "word", "number": 120, "path": ""},
        }
        cfg["recall"].update({
            "method": "MGMemoryRecall",
            "warning_threshold": 0.8,
            "warning_content": "[Warning: memory capacity is near the limit]",
            "recall_retrieval": make_text_retrieval_config(topk=topk, st_model=st_model),
            "archival_retrieval": make_text_retrieval_config(topk=topk, st_model=st_model),
            "trigger": {
                "method": "LLMTrigger",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "func_list": [
                    {"name": "memory_recall", "args": ["query"], "args_type": ["str"], "func_description": "Retrieve related items from recall storage into FIFO memory.", "args_description": ["query: retrieval query"]},
                    {"name": "memory_retrieval", "args": ["query"], "args_type": ["str"], "func_description": "Retrieve related items from archival storage into working memory.", "args_description": ["query: retrieval query"]},
                    {"name": "memory_transfer", "args": ["memory_list"], "args_type": ["list"], "func_description": "Transfer items from FIFO memory to working memory.", "args_description": ["memory_list: list of FIFO indexes"]},
                    {"name": "memory_archive", "args": ["memory_list"], "args_type": ["list"], "func_description": "Archive items from FIFO memory into recall storage.", "args_description": ["memory_list: list of FIFO indexes"]},
                    {"name": "memory_save", "args": ["memory_list"], "args_type": ["list"], "func_description": "Save items from working memory into archival storage.", "args_description": ["memory_list: list of working memory indexes"]},
                ],
                "few_shot": "",
                "prompt": {"template": "You are managing a memory OS.\n{warning_content}{no_execute_prompt}\n{function_prompt}\n\nMemory state:\n{memory_prompt}\n\nUser text:\n{text}\n\nReturn ONE OR MORE function calls, one per line (e.g. memory_recall(\"Alice\")).\nIf no function is needed, return: NO_EXECUTE", "input_variables": ["warning_content", "no_execute_prompt", "function_prompt", "few_shot", "memory_prompt", "text"]},
                "no_execuate": "NO_EXECUTE",
            },
        })
        return MGMemory(MemoryConfig(cfg))

    # --- RFMemory (Reflexion): FUMemoryStore + RFMemoryRecall + optional optimize ---
    def build_rf():
        cfg = make_common_config()
        cfg["name"] = "RFMemory"
        cfg["store"] = {"method": "FUMemoryStore"}
        cfg["recall"].update({"method": "RFMemoryRecall"})
        cfg["optimize"] = {
            "reflector": {
                "method": "TrialReflector",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "example": "Trial: I forgot the user's constraint.\nInsight: Always restate constraints before proposing actions.",
                "prompt": {"template": "You improve an agent by writing a single concise insight.\nPrevious insight: {previous_insight}\nNew trial:\n{new_trial}\nExample:\n{example}\nReturn ONLY the new insight.", "input_variables": ["previous_insight", "new_trial", "example"]},
            }
        }
        return RFMemory(MemoryConfig(cfg))

    # --- MTMemory (MemTree): tree storage + summarizer, recall via LTMemoryRecall ---
    def build_mt():
        cfg = make_common_config()
        cfg["name"] = "MTMemory"
        cfg["store"] = {
            "method": "MTMemoryStore",
            "traverse_base_threshold": 0.15,
            "traverse_rate": 1.0,
            "summarizer": {
                "method": "LLMSummarizer",
                "LLM_config": {"method": "APILLM", "name": JUDGE_LLM_CONFIG["name"], "api_key": JUDGE_LLM_CONFIG["api_key"], "base_url": JUDGE_LLM_CONFIG["base_url"], "temperature": 0.0},
                "prompt": {"template": "You are updating a parent node summary in a memory tree.\nCurrent content: {current_content}\nNew child content: {new_content}\nThere are {n_children} children.\nReturn an updated concise summary.", "input_variables": ["n_children", "new_content", "current_content"]},
            },
        }
        cfg["recall"].update({"method": "LTMemoryRecall", "text_retrieval": make_text_retrieval_config(topk=8, st_model=st_model)})
        return MTMemory(MemoryConfig(cfg))

    return [
        MemorySpec(name="FUMemory", build=build_fu, store_mode="text", recall_topk=topk),
        MemorySpec(name="STMemory", build=build_st, store_mode="text", recall_topk=topk),
        MemorySpec(name="LTMemory", build=build_lt, store_mode="text", recall_topk=topk),
        MemorySpec(name="MBMemory", build=build_mb, store_mode="mb", recall_topk=topk),
        MemorySpec(name="GAMemory", build=build_ga, store_mode="text", recall_topk=topk),
        MemorySpec(name="SCMemory", build=build_sc, store_mode="text", recall_topk=topk),
        MemorySpec(name="MGMemory", build=build_mg, store_mode="text", recall_topk=topk),
        MemorySpec(name="RFMemory", build=build_rf, store_mode="text", recall_topk=topk),
        MemorySpec(name="MTMemory", build=build_mt, store_mode="text", recall_topk=topk),
    ]


In [7]:
# --- Benchmark runner ---

import time

def evaluate_one_qa(*, question: str, gold: Any, adapter: MemoryAdapter, llm_cfg: dict) -> dict:
    gold_s = _normalize_text(gold)
    context = adapter.recall_context(question)
    pred = answer_question(question=question, context=context, llm_cfg=llm_cfg)

    judge = llm_judge(question=question, gold=gold_s, pred=pred, context=context, llm_cfg=JUDGE_LLM_CONFIG)

    return {
        "question": question,
        "gold": gold_s,
        "pred": pred,
        "context": context,
        "judge_score": judge.get("score", 0.0),
        "judge_verdict": judge.get("verdict", ""),
    }


def add_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # lexical metrics
    f1s, rls, b1s, b2s, mets = [], [], [], [], []
    for pred, gold in zip(df["pred"].tolist(), df["gold"].tolist()):
        f1s.append(token_f1(pred, gold))
        rls.append(rouge_l_f(pred, gold))
        b1, b2 = bleu_1_2(pred, gold)
        b1s.append(b1)
        b2s.append(b2)
        mets.append(meteor(pred, gold))

    df["F1"] = f1s
    df["RL"] = rls
    df["B1"] = b1s
    df["B2"] = b2s
    df["METEOR"] = mets

    # semantic metrics (batch)
    df["Sim"] = cosine_sim_batch(df["pred"].tolist(), df["gold"].tolist())
    df["BERTF1"] = bert_f1_batch(df["pred"].tolist(), df["gold"].tolist())

    return df


def run_locomo_benchmark(
    *,
    memory_spec: MemorySpec,
    llm_cfg: dict = ANSWER_LLM_CONFIG,
    top_samples: Optional[int] = None,
    top_qas_per_sample: Optional[int] = None,
    time_bucket: int = 20,
    show_progress: bool = True,
) -> pd.DataFrame:
    rows = []
    samples = locomo_raw[:top_samples] if top_samples else locomo_raw

    it = tqdm(samples, desc=f"{memory_spec.name} samples", disable=not show_progress)
    for sample in it:
        mem = memory_spec.build()
        adapter = MemoryAdapter(mem, store_mode=memory_spec.store_mode, time_bucket=time_bucket)

        turns = flatten_conversation(sample)
        adapter.reset()
        adapter.store_turns(turns)

        qa = sample.get("qa", [])
        qa_valid = [x for x in qa if "answer" in x and "question" in x]
        if top_qas_per_sample:
            qa_valid = qa_valid[:top_qas_per_sample]

        for qa_item in qa_valid:
            r = evaluate_one_qa(question=qa_item["question"], gold=qa_item["answer"], adapter=adapter, llm_cfg=llm_cfg)
            r.update({
                "sample_id": sample.get("sample_id"),
                "memory": memory_spec.name,
            })
            rows.append(r)

    df = pd.DataFrame(rows)
    if len(df):
        df = add_metrics(df)
    return df


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["F1", "RL", "B1", "B2", "METEOR", "BERTF1", "Sim", "judge_score"]
    out = (
        df.groupby("memory")[metric_cols]
        .mean()
        .sort_values("judge_score", ascending=False)
        .reset_index()
    )
    return out


In [8]:
# --- Run benchmark (small smoke test first) ---

# Warmup: pre-load the same SentenceTransformer model used by MBMemory/LTMemory.
# This avoids 5–10 min first-time download/load happening inside memory_spec.build().
# Run this cell once; then build() loads from cache and is much faster.
ST_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print("Warmup: loading SentenceTransformer (used by MBMemory recall)...")
_ = SentenceTransformer(ST_MODEL, device=DEVICE).encode(["warmup"], normalize_embeddings=True)
print("Warmup done. Building memory will now use cache.")

specs = make_memory_specs(topk=5, st_model=ST_MODEL)

# Smoke test settings (keep small to validate end-to-end quickly)
TOP_SAMPLES = 1
TOP_QAS = 3
# Limit turns so store phase does fewer summarization LLM calls (e.g. 60 turns → 3 buckets with time_bucket=20)
MAX_TURNS = 60

all_dfs = []
for spec in specs:
    df = run_locomo_benchmark(
        memory_spec=spec,
        top_samples=TOP_SAMPLES,
        top_qas_per_sample=TOP_QAS,
        # max_turns=MAX_TURNS,
        time_bucket=20,
        show_progress=True,
    )
    all_dfs.append(df)

df_small = pd.concat(all_dfs, ignore_index=True)
print("rows:", len(df_small))
display(summarize(df_small))

# Inspect a few examples
cols_show = ["memory","question","gold","pred","F1","RL","B1","B2","METEOR","BERTF1","Sim","judge_score","judge_verdict"]
display(df_small[cols_show].head(8))


Warmup: loading SentenceTransformer (used by MBMemory recall)...
Warmup done. Building memory will now use cache.


FUMemory samples: 100%|██████████| 1/1 [00:17<00:00, 17.93s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
STMemory samples: 100%|██████████| 1/1 [00:17<00:00, 17.42s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
LTMemory samples: 100%|██████████| 1/1 [00:23<00:00, 23.18s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predict

Successfully execute Function [memory_recall(['LGBTQ support group'])]
Successfully execute Function [memory_recall(['sunrise'])]
Successfully execute Function [memory_recall(["Caroline's education"])]


MGMemory samples: 100%|██████████| 1/1 [00:27<00:00, 27.68s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
RFMemory samples: 100%|██████████| 1/1 [00:13<00:00, 13.99s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
MTMemory samples: 100%|██████████| 1/1 [03:08<00:00, 188.58s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predic

rows: 27


,memory,F1,RL,B1,B2,METEOR,BERTF1,Sim,judge_score
0,LTMemory,0.111111,0.111111,0.111111,0.043033,0.055556,0.858926,0.461986,0.433333
1,SCMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.888965,0.534508,0.416667
2,GAMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.853692,0.451563,0.400000
3,MBMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.853692,0.451563,0.333333
4,MTMemory,0.000000,0.000000,0.000000,0.000000,0.000000,0.861203,0.315202,0.333333
5,FUMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.887629,0.413428,0.166667
6,MGMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.887629,0.413428,0.166667
7,RFMemory,0.095238,0.095238,0.083333,0.030429,0.053763,0.887629,0.413428,0.166667
8,STMemory,0.000000,0.000000,0.000000,0.000000,0.000000,0.839807,0.137673,0.000000


,memory,question,gold,pred,F1,RL,B1,B2,METEOR,BERTF1,Sim,judge_score,judge_verdict
0,FUMemory,When did Caroline go to the LGBTQ support group?,7 May 2023,Yesterday,0.000000,0.000000,0.00,0.000000,0.00000,0.837181,0.340239,0.0,incorrect
1,FUMemory,When did Melanie paint a sunrise?,2022,Unknown,0.000000,0.000000,0.00,0.000000,0.00000,0.992907,0.252833,0.0,unknown
2,FUMemory,What fields would Caroline be likely to pursue...,"Psychology, counseling certification",Counseling or mental health,0.285714,0.285714,0.25,0.091287,0.16129,0.832800,0.647211,0.5,partially_correct
3,STMemory,When did Caroline go to the LGBTQ support group?,7 May 2023,Unknown,0.000000,0.000000,0.00,0.000000,0.00000,0.834374,0.180657,0.0,unknown
4,STMemory,When did Melanie paint a sunrise?,2022,D1:14,0.000000,0.000000,0.00,0.000000,0.00000,0.834362,0.127853,0.0,incorrect
5,STMemory,What fields would Caroline be likely to pursue...,"Psychology, counseling certification",Unknown,0.000000,0.000000,0.00,0.000000,0.00000,0.850684,0.104507,0.0,unknown
6,LTMemory,When did Caroline go to the LGBTQ support group?,7 May 2023,Yesterday,0.000000,0.000000,0.00,0.000000,0.00000,0.837181,0.340239,0.0,incorrect
7,LTMemory,When did Melanie paint a sunrise?,2022,Last year,0.000000,0.000000,0.00,0.000000,0.00000,0.891096,0.367237,0.5,partially_correct
